# Agent Tool-Call Web Research

Give an agent an annotated search function so it can retrieve web results before writing its summary.

## 1. Install dependencies

In [ ]:
%pip install -q openai-agents ddgs

## 2. Choose a model provider

Set `PROVIDER` to `"openai"` or `"gemini"`, then add the corresponding API key to Colab Secrets.

In [ ]:
import os

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

PROVIDER = "gemini"  # Change to "openai" to use OpenAI.
OPENAI_MODEL_NAME = "gpt-5-mini"
GEMINI_MODEL_NAME = "gemini-2.5-flash"

if PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    if not os.environ["OPENAI_API_KEY"]:
        raise ValueError("Add OPENAI_API_KEY to Colab Secrets.")
    model = OPENAI_MODEL_NAME
elif PROVIDER == "gemini":
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("Add GEMINI_API_KEY to Colab Secrets.")
    set_tracing_disabled(disabled=True)
    model = OpenAIChatCompletionsModel(
        model=GEMINI_MODEL_NAME,
        openai_client=AsyncOpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=GEMINI_API_KEY,
        ),
    )
else:
    raise ValueError("PROVIDER must be 'openai' or 'gemini'.")

## 3. Define the agent tool

The `@function_tool` annotation makes `search_web` available for the agent to call.

In [ ]:
from agents import function_tool
from ddgs import DDGS
from ddgs.exceptions import DDGSException


@function_tool
def search_web(query: str) -> str:
    """Search the web and return source titles, summaries, and URLs."""
    try:
        results = DDGS(timeout=15).text(query, max_results=5)
    except DDGSException as error:
        print(f"Search failed: {error}")
        return "Search is temporarily unavailable."

    print(f"Search query: {query}")
    print(f"Results found: {len(results)}")
    for index, result in enumerate(results, start=1):
        print(f"\n{index}. {result.get('title', 'Untitled source')}")
        print(result.get('body', 'No summary available.'))
        print(result.get('href', 'No URL available.'))

    return "\n\n".join(
        f"{result.get('title', 'Untitled source')}\n{result.get('body', '')}\n{result.get('href', '')}"
        for result in results
    )


@function_tool
def print_completion_message() -> str:
    """Print a message when research is complete and the final answer is ready."""
    message = "Research complete. Preparing the final answer."
    print(message)
    return message

## 4. Let the agent search and summarize

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Research Assistant",
    instructions=(
        "Always call search_web before answering. After you have gathered the research, "
        "call print_completion_message immediately before your final answer. Summarize only "
        "the search results and end with a Sources section containing the URLs you used."
    ),
    tools=[search_web, print_completion_message],
    model=model,
)

question = "What is agentic AI, and how is it used in business?"
result = await Runner.run(starting_agent=agent, input=question)

print(result.final_output)